# Evaluation notebook for the final CatBoost model

This notebook loads the final trained CatBoost model and reproduces the evaluation metrics reported in the manuscript using the exact predefined hold-out split.

It is intended for inference and metric verification only; it does not reproduce hyperparameter optimization or model training.

In [1]:
# Evaluation notebook for the final CatBoost model
# This notebook loads the final trained CatBoost model and reproduces
# the evaluation metrics reported in the manuscript using the exact
# predefined hold-out split.
#
# It is intended for inference and metric verification only;
# it does not reproduce hyperparameter optimization or model training.

In [2]:
!pip -q install catboost openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.1 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
)

In [4]:
DATA_URL = "https://raw.githubusercontent.com/sarvelos/synthetic-bee-communities/main/merged_audio_features.xlsx"
MODEL_URL = "https://raw.githubusercontent.com/sarvelos/synthetic-bee-communities/model/catboost_fixed_subset_model.cbm"
LABELS_URL = "https://raw.githubusercontent.com/sarvelos/synthetic-bee-communities/model/label_classes.json"
TEST_IDX_URL = "https://raw.githubusercontent.com/sarvelos/synthetic-bee-communities/model/test_indices.json"

In [5]:
TARGET = "Specie"

FEATURES = [
    "mfcc_1",
    "spectral_contrast",
    "mfcc_5",
    "mfcc_10",
    "mfcc_4",
    "flatness",
    "f0_yin_mean",
    "mfcc_7",
    "mfcc_13",
    "rolloff",
    "mfcc_11",
    "delta_1",
    "mfcc_8",
    "mfcc_sd_9",
    "mfcc_sd_5",
    "Type",
]

df = pd.read_excel(DATA_URL)

missing_cols = [c for c in [TARGET] + FEATURES if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

X = df[FEATURES].copy()
X["Type"] = X["Type"].astype("category")

print("Dataset shape:", df.shape)
print("Number of species:", df[TARGET].nunique())

Dataset shape: (396, 80)
Number of species: 11


In [6]:
class_labels = pd.read_json(LABELS_URL, typ="series").tolist()
test_indices = pd.read_json(TEST_IDX_URL, typ="series").tolist()

le = LabelEncoder()
le.fit(class_labels)

y = le.transform(df[TARGET].astype(str))

# Reconstruct exact hold-out split used in the manuscript
test_indices = list(map(int, test_indices))
all_indices = list(range(len(df)))
train_indices = sorted(set(all_indices) - set(test_indices))

X_train = X.loc[train_indices].copy()
y_train = y[np.array(train_indices)]

X_test = X.loc[test_indices].copy()
y_test = y[np.array(test_indices)]

print("Recovered train shape:", X_train.shape)
print("Recovered test shape:", X_test.shape)

Recovered train shape: (316, 16)
Recovered test shape: (80, 16)


In [7]:
!wget -q -O catboost_fixed_subset_model.cbm "$MODEL_URL"

model = CatBoostClassifier()
model.load_model("catboost_fixed_subset_model.cbm")

print("Model loaded successfully.")

Model loaded successfully.


In [9]:
pred_train = np.array(model.predict(X_train)).astype(int).ravel()
pred_test = np.array(model.predict(X_test)).astype(int).ravel()

print("Train predictions shape:", pred_train.shape)
print("Test predictions shape:", pred_test.shape)

Train predictions shape: (316,)
Test predictions shape: (80,)


In [10]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

train_metrics = compute_metrics(y_train, pred_train)
test_metrics = compute_metrics(y_test, pred_test)

metrics_df = pd.DataFrame([train_metrics, test_metrics], index=["train", "test"]).round(4)
metrics_df

,accuracy,balanced_accuracy,macro_f1,macro_precision,macro_recall,mcc
train,1.0,1.0000,1.0000,1.0000,1.0000,1.0000
test,0.8,0.7971,0.7959,0.8317,0.7971,0.7829


In [11]:
print("Train metrics:")
for k, v in train_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

Train metrics:
accuracy: 1.0000
balanced_accuracy: 1.0000
macro_f1: 1.0000
macro_precision: 1.0000
macro_recall: 1.0000
mcc: 1.0000

Test metrics:
accuracy: 0.8000
balanced_accuracy: 0.7971
macro_f1: 0.7959
macro_precision: 0.8317
macro_recall: 0.7971
mcc: 0.7829


In [12]:
print(classification_report(y_test, pred_test, target_names=le.classes_, digits=4))

                       precision    recall  f1-score   support

         Bombus morio     0.7500    0.8571    0.8000         7
    Bombus pauloensis     1.0000    0.4286    0.6000         7
      Centris fuscata     0.8333    0.7143    0.7692         7
      Eulaema nigrita     0.8889    1.0000    0.9412         8
      Exomalopsis sp.     0.7778    1.0000    0.8750         7
          Melipona sp     1.0000    1.0000    1.0000         7
     Oxaea flavescens     0.5833    0.8750    0.7000         8
    Piltiloglossa sp.     0.8571    0.7500    0.8000         8
Xylocopa hirsutissima     1.0000    0.7143    0.8333         7
          Xylocopa sp     0.6250    0.7143    0.6667         7
    Xylocopa suspecta     0.8333    0.7143    0.7692         7

             accuracy                         0.8000        80
            macro avg     0.8317    0.7971    0.7959        80
         weighted avg     0.8296    0.8000    0.7965        80



In [13]:
cm = confusion_matrix(y_test, pred_test)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
cm_df

,Bombus morio,Bombus pauloensis,Centris fuscata,Eulaema nigrita,Exomalopsis sp.,Melipona sp,Oxaea flavescens,Piltiloglossa sp.,Xylocopa hirsutissima,Xylocopa sp,Xylocopa suspecta
Bombus morio,6,0,0,1,0,0,0,0,0,0,0
Bombus pauloensis,0,3,0,0,1,0,3,0,0,0,0
Centris fuscata,0,0,5,0,0,0,0,0,0,2,0
Eulaema nigrita,0,0,0,8,0,0,0,0,0,0,0
Exomalopsis sp.,0,0,0,0,7,0,0,0,0,0,0
Melipona sp,0,0,0,0,0,7,0,0,0,0,0
Oxaea flavescens,0,0,0,0,1,0,7,0,0,0,0
Piltiloglossa sp.,0,0,0,0,0,0,1,6,0,0,1
Xylocopa hirsutissima,1,0,0,0,0,0,0,0,5,1,0
Xylocopa sp,1,0,1,0,0,0,0,0,0,5,0
